# 🏆 ESPN Fantasy with `sportsdataverse-py`

ESPN runs five fantasy games on one platform: football (`ffl`), men's basketball (`fba`), baseball (`flb`), hockey (`fhl`) and women's basketball (`wfba`). `sportsdataverse.providers.espn_fantasy` reads the parts of it that are **public**: the games and seasons, the pro-team schedules, the player universe, ESPN's default scoring, player news, and the public leagues ESPN lists in its draft lobby.

No account and no cookies. A private league answers HTTP 401 and is out of scope.

## 🧰 The toolbox

| Function | What it gives you |
|---|---|
| [`espn_fantasy_games`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_games) | Every fantasy game and its current season |
| [`espn_fantasy_players`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_players) | The player universe of a season, with ownership |
| [`espn_fantasy_pro_schedule`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_pro_schedule) | Pro games by scoring period |
| [`espn_fantasy_default_scoring`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_default_scoring) | ESPN's default points per stat |
| [`espn_fantasy_league_directory`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_league_directory) | ESPN's public leagues |
| [`espn_fantasy_league_teams`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_league_teams) | A public league's standings |
| [`espn_fantasy_league_matchups`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_league_matchups) | Its head-to-head schedule |
| [`espn_fantasy_player_pool`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_player_pool) | Its free agents and rostered players |
| [`espn_fantasy_player_news`](../espn_fantasy/reference/espn_fantasy.md#espn_fantasy_player_news) | News items for one player |

## 🔌 Setup

```sh
pip install --pre "sportsdataverse"
```

`--pre` is needed while 0.2.0 is in beta.

In [ ]:
import polars as pl

from sportsdataverse.providers import espn_fantasy as ef

pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)

Fantasy routes are seasonal: an off-season game lists no public leagues, and a league that is still drafting has empty rosters. A small `safe()` helper prints a one-line skip instead of stopping the notebook, naming whether the answer was *nothing there* (`NoDataError`) or *the fetch failed* (`AssetFetchError`). 🛟

In [ ]:
from sportsdataverse.errors import AssetFetchError, NoDataError

def safe(label, thunk):
    try:
        out = thunk()
        print(f"✅ {label}")
        return out
    except (NoDataError, AssetFetchError) as e:
        print(f"\u23ed\ufe0f  {label}: {type(e).__name__}: {e}")
        return None

## 🎮 Games and seasons

Season ids are ESPN's own: football is named by the year it **starts** (`2026` = fall 2026), basketball and hockey by the year they **end** (2026-27 = `2027`). `espn_fantasy_games` gives each game's current one.

In [ ]:
games = ef.espn_fantasy_games()
season = dict(games.select(pl.col("game_abbrev").str.to_lowercase(), "current_season_id").iter_rows())
games.select("game_abbrev", "game_name", "pro_sport_abbrev", "current_season_id", "current_scoring_period_id")

## 🧑‍🤝‍🧑 The player universe

`espn_fantasy_players` returns every active player of a season with the share of ESPN leagues that roster them. `player_id` is ESPN's athlete id, the same one ESPN's other APIs use.

In [ ]:
players = ef.espn_fantasy_players("fba", season["fba"])
players.sort("percent_owned", descending=True).select("player_id", "full_name", "pro_team_id", "percent_owned").head(10)

## 📅 Pro schedule and scoring

Football's scoring period is the NFL week; for the other games it is a day. The pro schedule says which pro games count toward each period, and ESPN's default template says what each stat is worth.

In [ ]:
sched = ef.espn_fantasy_pro_schedule("ffl", season["ffl"])
sched.group_by("scoring_period_id").agg(pl.len().alias("pro_games")).sort("scoring_period_id").head(6)

In [ ]:
scoring = ef.espn_fantasy_default_scoring("ffl", season["ffl"], league_default_id=1)
scoring.select("scoring_type", "stat_id", "points").sort("points", descending=True).head(8)

## 🏟️ A public league

ESPN's draft lobby lists public leagues anyone can join; every one of them is readable by id. Leagues whose draft is over have rosters and played matchups.

In [ ]:
lobby = ef.espn_fantasy_league_directory("fba", season["fba"])
done = lobby.filter(pl.col("full") == True).sort("draft_date")
done.select("league_id", "league_name", "scoring_type", "draft_type", "draft_date").head(5)

In [ ]:
league_id = done["league_id"][0] if done.height else None
teams = safe("league teams", lambda: ef.espn_fantasy_league_teams("fba", season["fba"], league_id)) if league_id else None
teams.select("team_id", "wins", "losses", "points_for", "playoff_seed").sort("playoff_seed") if teams is not None else None

In [ ]:
matchups = safe("league matchups", lambda: ef.espn_fantasy_league_matchups("fba", season["fba"], league_id)) if league_id else None
matchups.filter(pl.col("winner") != "UNDECIDED").tail(5) if matchups is not None else None

The player pool answers *who is still available?* The `X-Fantasy-Filter` header narrows it: here, the ten most-owned free agents.

In [ ]:
fa_filter = '{"players":{"limit":10,"sortPercOwned":{"sortPriority":1,"sortAsc":false},'
fa_filter += '"filterStatus":{"value":["FREEAGENT","WAIVERS"]}}}'
pool = safe("free agents", lambda: ef.espn_fantasy_player_pool("fba", season["fba"], league_id, fantasy_filter=fa_filter)) if league_id else None
pool.select("player_id", "full_name", "status", "percent_owned") if pool is not None else None

## 📰 Player news

In [ ]:
top = players.sort("percent_owned", descending=True)["player_id"][0]
news = safe("player news", lambda: ef.espn_fantasy_player_news("fba", top))
news.select("published", "type", "headline").head(5) if news is not None else None

## 🧭 Where next

- Every wrapper takes `return_parsed=False` for the raw body and `return_as_pandas=True` for pandas.
- `team_id` is unique only inside one league; join league tables on `(league_id, team_id)`.
- Private leagues need their owner's `espn_s2` / `SWID` cookies; this package does not take them.
- The full list is on the [reference page](../espn_fantasy/reference/espn_fantasy.md).